## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [8]:
data = pd.read_csv('/Users/olesialev/Downloads/statistical_hypothesis/cookie_cats.csv')
data.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [9]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [10]:
#1.Дані за умовою задачі
base_retention = 0.19              
expected_retention = 0.19 + 0.01  
#2. Розрахунок розміру ефекту 
effect_size = sms.proportion_effectsize(0.19, 0.20)

In [12]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )                                                 
required_n = ceil(required_n)                          
print(required_n)

print(f"Необхідна кількість спостережень для кожної групи: {required_n}")
print(f"Загальна кількість користувачів для тесту: {required_n * 2}")

24638
Необхідна кількість спостережень для кожної групи: 24638
Загальна кількість користувачів для тесту: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [14]:
import pandas as pd

df = data

# 2. Обчислюємо середнє значення для кожної групи
retention_by_version = df.groupby('version')['retention_7'].mean()

print("Середнє утримання на 7-й день за версіями гри:")
print(retention_by_version)

diff = (retention_by_version['gate_30'] - retention_by_version['gate_40']) / retention_by_version['gate_40'] * 100
print(f"Різниця: gate_30 краще за gate_40 на {diff:.2f}%")

Середнє утримання на 7-й день за версіями гри:
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64
Різниця: gate_30 краще за gate_40 на 4.51%


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [15]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

#1. Дані для тесту
control_results = df[df['version'] == 'gate_30']['retention_7']
treatment_results = df[df['version'] == 'gate_40']['retention_7']

n_con = control_results.count()
n_treat = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_con, n_treat]

#2. z-тест та довірчі інтервали
z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


1.Так, різниця є статистично значущою.
Оскільки отримане p-value є меншим за рівень значущості alpha, ми відхиляємо нульову гіпотезу, що свідчить про реальний негативний вплив переміщення воріт на рівень утримання гравців.

2.Довірчі інтервали не перетинаються, що підтверджує надійність наших даних і вказує на стабільну перевагу версії gate_30.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [17]:
from scipy.stats import chi2_contingency

# Створюємо таблицю спряженості
contingency_table = pd.crosstab(df['version'], df['retention_7'])

# Проводимо тест Хі-квадрат
chi2, p_val, dof, expected = chi2_contingency(contingency_table)

print(f"χ² = {chi2:.3f}")
print(f"p-value: {p_val:.5f}")

χ² = 9.959
p-value: 0.00160


Оскільки $p < 0.05$, ми відхиляємо $H_0$. Існує статистично значуща залежність: переміщення воріт суттєво впливає на те, чи повертаються гравці через тиждень.